# N21 · Memory Snapshot 走读：从 stack 定位泄露

> 配套 `labs/l02.5_memory_snapshot/`。CPU 模拟，思路与 `torch.cuda.memory._dump_snapshot` 一致。
>
> 读完后你应能：
> 1. 解释 caching allocator 为什么让 `nvidia-smi` 不可信
> 2. 用 stack 聚合定位"哪行代码累计 leak 了多少 GB"
> 3. 区分"真泄露"与"caching allocator 的 fragmentation"


## 1. 真实事故场景

SGLang VLM RL 训练，跑到 80 步突然 OOM：
- `nvidia-smi`：显示 78GB / 80GB used
- `torch.cuda.memory_summary()`：active=12GB, reserved=78GB（caching allocator 缓存了大量未归还内存）
- 调小 batch size：还是 OOM

问题不是"前向后向爆炸"，而是 forward hook 在每个 step 持有了一份 image tensor 引用，累计 80 step × 800MB = 64GB。


In [ ]:
# 模拟一个有 hook 泄露的训练 loop
import inspect
from dataclasses import dataclass

@dataclass
class AllocEvent:
    addr: int
    size: int
    stack: tuple

class Tracker:
    def __init__(self):
        self.live = {}
        self.next = 1
    def alloc(self, size, stack):
        addr = self.next
        self.next += size + 64
        self.live[addr] = AllocEvent(addr, size, stack)
        return addr
    def free(self, addr):
        self.live.pop(addr, None)

def get_stack():
    return tuple(f"{f.function}:{f.lineno}" for f in inspect.stack()[1:5])

tr = Tracker()

def forward(x):
    # 正常分配，结束后释放
    addr = tr.alloc(1024 * 1024, get_stack())  # 1MB activation
    tr.free(addr)
    return x

leak_jar = []  # 模拟 hook 持有的引用

def forward_with_leaky_hook(x):
    addr = tr.alloc(800 * 1024 * 1024, get_stack())  # 800MB image tensor
    leak_jar.append(addr)  # ← 泄露！addr 不会被 free
    return x

for step in range(10):
    forward("x")
    forward_with_leaky_hook("x")

print(f"after 10 steps: live={len(tr.live)} allocations, total={sum(e.size for e in tr.live.values()) / 1e9:.2f} GB")

In [ ]:
# 按 stack 聚合，立刻看出泄露源
from collections import defaultdict

by_stack = defaultdict(int)
for ev in tr.live.values():
    by_stack[ev.stack] += ev.size

top = sorted(by_stack.items(), key=lambda kv: -kv[1])[:3]
for stack, total in top:
    print(f"{total / 1e9:.2f} GB  from")
    for frame in stack[:3]:
        print(f"   {frame}")
    print()

**关键洞察**：泄露的 stack 里包含 `forward_with_leaky_hook`，3 秒就定位了。如果只看 `nvidia-smi` 不知道哪行代码引起的。

真实 PyTorch 流程：
```python
torch.cuda.memory._record_memory_history(max_entries=100000)
# ... 跑 N 个 step ...
torch.cuda.memory._dump_snapshot("./snap.pickle")
# 然后用 https://pytorch.org/memory_viz 可视化或自己脚本聚合
```


## 2. caching allocator 与 fragmentation

注意：CUDA caching allocator 不会立刻把 free 还给 driver——`reserved` 可能远大于 `active`。
看到 `reserved` 增长但 `active` 稳定，多半是 fragmentation 问题，不是真泄露。判别法：

- **真泄露**：`active` 增长，`live_allocations` 数量增长，stack 聚合能找到来源
- **Fragmentation**：`reserved` 增长但 `active` 稳定，需要 `expandable_segments` 或调整 alloc pattern

## 3. 自检 / 面试题

1. 为什么 `nvidia-smi` 显示 80GB 占满，但程序 `torch.cuda.memory_allocated()` 只有 12GB？（caching allocator reserved）
2. memory snapshot 的 stack 信息从哪来？（`torch.cuda.memory._record_memory_history` 注册了 allocator hook，每次 alloc 都 walk Python stack）
3. 看到 OOM stack 都指向 `flash_attn` kernel 时，到底是 attention kernel 真的 OOM 还是别的代码先把显存吃光？（snapshot 里 alloc 的时间序列才能告诉你；纯看最后一帧不可靠）
